# CVRP Comparison: Classical Sweep vs. Quantum-Hybrid QAOA

This notebook compares the classical proximity-clustering solver (`yale_output`) against the QAOA hybrid solver (`hybrid_output`) across all four CVRP instances.

**Sections:**
1. Hardcoded results from both runs (no re-execution needed)
2. Feasibility validation — do all routes satisfy the CVRP constraints?
3. Brute-force optimal — enumerate every valid partition to get ground truth
4. Approximation ratios — how close is each solver to optimal?
5. Scalability analysis — how do QAOA resources grow with problem size?
6. Discussion — honest assessment of quantum advantage

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from itertools import permutations, combinations

## Shared Utilities

Identical to both solvers — distance matrix and route cost.

In [ ]:
def euclidean(p1, p2):
    return math.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)

def build_distance_matrix(locations):
    ids = list(locations.keys())
    return {(i,j): euclidean(locations[i], locations[j]) for i in ids for j in ids if i != j}

def encode_locations(depot, customers):
    locs = {0: tuple(depot)}
    for i, c in enumerate(customers, start=1):
        locs[i] = tuple(c)
    return locs

def route_distance(route, dist_matrix):
    return sum(dist_matrix[(route[k], route[k+1])] for k in range(len(route)-1))

def total_distance(routes, dist_matrix):
    return sum(route_distance(r, dist_matrix) for r in routes)

def build_route(group, dist_matrix):
    """Exact shortest round-trip via exhaustive permutation."""
    best, best_d = None, float('inf')
    for perm in permutations(group):
        route = [0] + list(perm) + [0]
        d = route_distance(route, dist_matrix)
        if d < best_d:
            best_d, best = d, route
    return best

## Section 1 — Hardcoded Results

Routes and total distances taken directly from executed notebook outputs. No re-execution.

**Note on Instance 3 quantum result:** QAOA ran on `ibm_fez` and converged (`energy = −67.12`). The selected bitstring placed both cuts at gaps 0 and 1 (the two cheapest detour costs), producing groups of size 1, 1, 4. This violates the capacity constraint C=2. See Section 2 for full feasibility analysis.

In [ ]:
depot = (0, 0)

instances = {
    1: {"customers": [(-2,2),(-5,8),(2,3)],                                                                                "N": 2, "C": 5},
    2: {"customers": [(-2,2),(-5,8),(2,3)],                                                                                "N": 2, "C": 2},
    3: {"customers": [(-2,2),(-5,8),(2,3),(5,7),(2,4),(2,-3)],                                                             "N": 3, "C": 2},
    4: {"customers": [(-2,2),(-5,8),(6,3),(4,4),(3,2),(0,2),(-2,3),(-4,3),(2,3),(2,7),(-2,5),(-1,4)], "N": 4, "C": 3},
}

# Routes from yale_output.ipynb execution
classical_routes = {
    1: [[0,3,2,1,0]],
    2: [[0,2,1,0], [0,3,0]],
    3: [[0,6,4,0], [0,3,5,0], [0,2,1,0]],
    4: [[0,5,3,4,0], [0,9,10,6,0], [0,12,11,2,0], [0,7,8,1,0]],
}

# Routes from hybrid_output.ipynb execution
# Instance 1: QAOA skipped (G=1), Instance 2: ibm_fez (not converged),
# Instance 3: ibm_fez (converged but INFEASIBLE — see Section 2),
# Instance 4: local AerSimulator, reps=2 (output not captured — excluded below)
quantum_routes = {
    1: [[0,3,2,1,0]],
    2: [[0,3,0], [0,2,1,0]],
    3: [[0,6,0], [0,4,0], [0,3,5,2,1,0]],   # INFEASIBLE
    4: None,                                   # Not captured
}

# Resource info from hybrid_output execution
quantum_resources = {
    1: {"qubits": 0,  "gates": 0,   "depth": 0,   "backend": "none (G=1)",          "time_s": 0.0},
    2: {"qubits": 2,  "gates": 31,  "depth": 16,  "backend": "ibm_fez",             "time_s": None},
    3: {"qubits": 5,  "gates": 333, "depth": 190, "backend": "ibm_fez",             "time_s": None},
    4: {"qubits": 11, "gates": None,"depth": None,"backend": "aer_simulator (local)","time_s": None},
}

## Section 2 — Feasibility Validation

A valid CVRP solution must satisfy three constraints:
1. Every route starts and ends at the depot (node 0).
2. Every customer is visited exactly once across all routes.
3. No route visits more than C customers.

**Why can our QUBO produce infeasible solutions?**
The QUBO penalises having the wrong *number* of cuts (≠ G−1), but says nothing about the *resulting group sizes*. For Instance 3 (H=6, G=3, C=2), placing both cuts at the two cheapest gaps (positions 0 and 1 in angular order) gives groups of size 1, 1, 4 — satisfying the cut-count constraint but violating capacity.

In [ ]:
def check_feasibility(routes, H, C):
    """Returns (is_feasible, list_of_violations)."""
    violations = []
    visited = []
    for i, route in enumerate(routes, start=1):
        if route[0] != 0 or route[-1] != 0:
            violations.append(f"r{i}: does not start/end at depot")
        custs = [n for n in route if n != 0]
        if len(custs) > C:
            violations.append(f"r{i}: {len(custs)} customers visited, capacity is {C}")
        visited.extend(custs)
    expected = list(range(1, H+1))
    if sorted(visited) != expected:
        extra   = sorted(set(visited) - set(expected))
        missing = sorted(set(expected) - set(visited))
        if extra:   violations.append(f"Extra visits: {extra}")
        if missing: violations.append(f"Missing customers: {missing}")
    return len(violations) == 0, violations

print("=" * 60)
print(f"{'':>3}  {'Solver':<10}  {'Feasible':<10}  Violations")
print("=" * 60)
for inst, params in instances.items():
    H = len(params["customers"])
    C = params["C"]
    for label, routes in [("Classical", classical_routes[inst]),
                          ("Quantum",   quantum_routes[inst])]:
        if routes is None:
            print(f"  {inst}  {label:<10}  {'N/A':<10}  (not captured)")
            continue
        ok, issues = check_feasibility(routes, H, C)
        status = "YES" if ok else "NO"
        detail = "—" if ok else "; ".join(issues)
        print(f"  {inst}  {label:<10}  {status:<10}  {detail}")

## Section 3 — Brute-Force Optimal

For each instance we enumerate **every valid partition** of customers into G groups (each of size ≤ C), compute the exact optimal route ordering within each group via permutations, and record the minimum total distance.

This gives the ground truth against which both solvers are measured.

| Instance | Valid partitions to check |
| :-: | :-: |
| 1 | 1 (G=1, single group) |
| 2 | 3 (partition 3 customers into 2 groups of ≤ 2) |
| 3 | 15 (partition 6 customers into 3 pairs) |
| 4 | 15,400 (partition 12 customers into 4 triples) |

In [ ]:
def all_partitions(items, k, max_size):
    """
    Yield all ways to partition `items` into exactly k non-empty
    ordered tuples, each of length ≤ max_size.
    Groups are generated in canonical order (first element of each
    group is the smallest) to avoid counting the same partition twice.
    """
    if k == 1:
        if 1 <= len(items) <= max_size:
            yield (tuple(items),)
        return
    first = items[0]
    rest  = items[1:]
    for size in range(1, min(max_size, len(items) - k + 1) + 1):
        for tail in combinations(rest, size - 1):
            group     = (first,) + tail
            remaining = [x for x in rest if x not in set(tail)]
            for sub in all_partitions(remaining, k - 1, max_size):
                yield (group,) + sub

def brute_force_optimal(depot, customers, N, C):
    locs       = encode_locations(depot, customers)
    dist_matrix = build_distance_matrix(locs)
    H = len(customers)
    G = min(math.ceil(H/C), N)
    cust_ids = list(range(1, H+1))

    best_dist   = float('inf')
    best_routes = None
    count = 0

    for partition in all_partitions(cust_ids, G, C):
        count += 1
        routes = [build_route(list(g), dist_matrix) for g in partition]
        dist   = total_distance(routes, dist_matrix)
        if dist < best_dist:
            best_dist   = dist
            best_routes = routes

    return best_routes, best_dist, count

print("Computing brute-force optimal for all instances...")
optimal = {}
for inst, params in instances.items():
    routes, dist, n_partitions = brute_force_optimal(depot, params["customers"], params["N"], params["C"])
    optimal[inst] = {"routes": routes, "dist": dist, "n_partitions": n_partitions}
    print(f"  Instance {inst}: optimal = {dist:.4f}  ({n_partitions} partitions checked)")
    for i, r in enumerate(routes, 1):
        print(f"    r{i}: {' -> '.join(str(n) for n in r)}")

## Section 4 — Approximation Ratios

$$\text{Approximation ratio} = \frac{d_{\text{algorithm}}}{d_{\text{optimal}}}$$

A ratio of 1.000 means the algorithm found the globally optimal solution. Values > 1 indicate suboptimality.

For infeasible quantum solutions (constraint violated), we report both the raw distance and label the solution `INFEASIBLE`. The approximation ratio is set to `∞` for infeasible results.

In [ ]:
def compute_dist_from_routes(routes_list, depot, customers):
    """Compute total distance given a list of route lists."""
    locs        = encode_locations(depot, customers)
    dist_matrix = build_distance_matrix(locs)
    return total_distance(routes_list, dist_matrix)

print(f"{'Inst':<6} {'Optimal':>10} {'Classical':>12} {'Ratio_C':>9} {'Quantum':>12} {'Ratio_Q':>9} {'Q feasible?':>12}")
print("-" * 75)

for inst, params in instances.items():
    H  = len(params["customers"])
    C  = params["C"]
    opt_dist = optimal[inst]["dist"]

    # Classical
    c_dist = compute_dist_from_routes(classical_routes[inst], depot, params["customers"])
    c_ratio = c_dist / opt_dist

    # Quantum
    if quantum_routes[inst] is None:
        q_dist_str  = "N/A"
        q_ratio_str = "N/A"
        q_feasible  = "N/A"
    else:
        q_routes = quantum_routes[inst]
        q_dist   = compute_dist_from_routes(q_routes, depot, params["customers"])
        ok, _    = check_feasibility(q_routes, H, C)
        q_dist_str  = f"{q_dist:.4f}"
        q_ratio_str = f"{q_dist/opt_dist:.4f}" if ok else "∞ (infeasible)"
        q_feasible  = "YES" if ok else "NO"

    print(f"{inst:<6} {opt_dist:>10.4f} {c_dist:>12.4f} {c_ratio:>9.4f} {q_dist_str:>12} {q_ratio_str:>9} {q_feasible:>12}")

## Section 5 — Scalability Analysis

### QAOA Resource Scaling

In our cut-point formulation, the QAOA circuit for H customers has:

| Resource | Formula | Reason |
|---|---|---|
| Qubits | $n = H - 1$ | One binary variable per angular gap |
| Z terms | $n$ | One per qubit (linear cost + constraint bias) |
| ZZ terms | $\binom{n}{2} = \frac{n(n-1)}{2}$ | All-pairs from constraint penalty $(\sum b_i - (G-1))^2$ |
| Gates per rep | $\approx 3 \times \binom{n}{2} + n$ | Each ZZ term → CNOT + Rz + CNOT; each Z term → Rz |
| Circuit depth (serialised) | $O(n^2)$ | ZZ gates serialised per Trotterisation layer |
| COBYLA parameters | $2 \times \text{reps}$ | Independent of $H$ |

The **quadratic growth** of ZZ terms (and thus gates) is the dominant cost. It arises because the constraint $(\sum b_i - (G-1))^2$ couples every pair of variables. This could be reduced to $O(n)$ by replacing the global penalty with a nearest-neighbour chain penalty — at the cost of a weaker constraint enforcement.

### Classical Sweep Scaling

| Step | Complexity |
|---|---|
| Angular sort | $O(H \log H)$ |
| Rotation search (H rotations × G groups × nearest-neighbour estimate) | $O(H^2)$ |
| Route building (permutations within each group, G groups) | $O(G \cdot C!)$ |

Both algorithms are $O(H^2)$ in the number of customers, but QAOA's constant is much larger (circuit simulation cost per iteration × number of COBYLA iterations).

In [ ]:
H_values = list(range(3, 22))

qubits      = [H - 1 for H in H_values]
zz_terms    = [n*(n-1)//2 for n in qubits]
total_gates = [3*zz + n for n, zz in zip(qubits, zz_terms)]  # per rep

# Actual observed data points from our runs
obs_H      = [2, 5]         # Instance 2 (n=2), Instance 3 (n=5)
obs_gates  = [31, 333]      # from transpiled gate counts (reps=2)
obs_depth  = [16, 190]

print(f"{'H':>4} {'Qubits':>8} {'ZZ terms':>10} {'Gates/rep':>12} {'Cum. gates (reps=2)':>22}")
print("-" * 60)
for H, n, zz, g in zip(H_values, qubits, zz_terms, total_gates):
    print(f"{H:>4} {n:>8} {zz:>10} {g:>12} {2*g:>22}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(H_values, qubits, 'b-o', markersize=4, label='Qubits (H−1)')
axes[0].plot(H_values, zz_terms, 'r-s', markersize=4, label='ZZ terms (quadratic)')
axes[0].set_xlabel('Number of customers H')
axes[0].set_ylabel('Count')
axes[0].set_title('QAOA Resources vs. Problem Size')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(H_values, total_gates, 'g-^', markersize=4, label='Gates per rep (O(H²))')
axes[1].scatter([H_values[obs_H[0]-3], H_values[obs_H[1]-3]],
                [obs_gates[0]//2, obs_gates[1]//2],
                color='black', zorder=5, label='Observed (transpiled, ÷2 for reps=2)')
axes[1].set_xlabel('Number of customers H')
axes[1].set_ylabel('Gate count')
axes[1].set_title('Circuit Gate Count vs. Problem Size')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('scalability.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved scalability.png")

## Section 6 — Discussion

### Solution Quality

Instances 1 and 2 are trivial: both solvers find the global optimum. Instance 1 uses a single vehicle (G=1), so QAOA is never invoked. Instance 2 has n=2 qubits and only 3 valid partitions — classical and quantum agree.

Instance 3 is where the interesting divergence appears:
- The **classical sweep** finds the only feasible consecutive-arc partition (cuts at gaps 1,3 giving groups [6,4],[3,5],[2,1]) with distance **50.696**.
- The **QAOA** (on ibm_fez) selected cuts at gaps 0,1 — the two cheapest cut-weight positions — giving groups of size 1,1,4. This is **infeasible** (violates C=2).
- The **true optimal** (from brute force) pairs customer 6 with customer 3, customers 5 with 4, and customers 2 with 1, achieving distance **≈49.50** — a solution that is not reachable by either solver because it requires pairing non-angularly-consecutive customers.

### Root Cause of Infeasibility (Instance 3)

The QUBO penalty $\lambda(\sum_i b_i - (G-1))^2$ enforces only that exactly $G-1$ cuts exist. It does **not** prevent all cuts from concentrating at one end of the angular order, which leaves one group too large. For Instance 3, only 1 of the $\binom{5}{2}=10$ possible 2-cut placements is feasible. The QAOA penalty was not strong enough to steer the optimiser away from the 9 infeasible placements.

**Fix:** Add a per-group capacity penalty to the QUBO:
$$\mu \sum_{g} \max(0,\, |\text{group}_g| - C)^2$$
or equivalently, constrain the problem to the Dicke subspace $|D(n, G-1)\rangle$ using an XY mixer, which strictly preserves the cut-count constraint.

### Does QAOA Offer Quantum Advantage Here?

For the **consecutive cut-point formulation** specifically:
- The search space is $\binom{H-1}{G-1}$ possible cut placements — polynomial in H for fixed G.
- The classical sweep in `yale_output` only explores H rotations (a strict subset), so it is **not** an exhaustive classical baseline. A classical exhaustive search of all $\binom{H-1}{G-1}$ placements would match or beat QAOA quality in the same O(H²) time.
- QAOA reps=1 finds COBYLA-optimal parameters in ~100–150 iterations; there is no known quantum speedup over this for polynomial-size search spaces.

**Where quantum can help:** if the formulation is extended to search over *all* partitions (not just consecutive arcs), the search space grows exponentially with H. In that regime, QAOA operating on a richer QUBO (G·H binary variables encoding full cluster membership) could in principle explore more of the landscape per query than a greedy classical heuristic — though proven quantum advantage for combinatorial optimisation remains an open research question.

### Summary Table

| | Instance 1 | Instance 2 | Instance 3 | Instance 4 |
|---|---|---|---|---|
| Classical distance | 21.74 | 26.18 | 50.70 | 59.54 |
| Quantum distance | 21.74 | 26.18 | 46.62* | — |
| Optimal distance | 21.74 | 26.18 | ~49.50 | (see brute force) |
| Classical approx. ratio | 1.000 | 1.000 | ~1.024 | — |
| Quantum approx. ratio | 1.000 | 1.000 | ∞ (infeasible)* | — |
| QAOA qubits | 0 | 2 | 5 | 11 |
| Transpiled gates | — | 31 | 333 | — |
| Backend | — | ibm_fez | ibm_fez | aer (local) |

*Quantum Instance 3 route violates C=2 capacity constraint. The valid quantum solution under the current formulation is also 50.70 (same as classical).